# 02.1 - Model experiments: CNN vs Transformer demo

Notebook nay dung dataset 9-channel tu notebook 01:

`ref one-hot 4 channels + alt one-hot 4 channels + mutation marker 1 channel`

Muc tieu: test nhanh nhieu huong model tren cung split:

- `cnn_baseline`: CNN 1D dang dung o notebook 02
- `cnn_residual`: CNN residual nho
- `transformer_small`: Transformer encoder demo cho sequence length 201

Doi model tai bien `MODEL_NAME`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path("/mnt/MyData/Bioinformatics/Project")
PROCESSED_DIR = PROJECT_DIR / "processed"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

X_PATH = PROCESSED_DIR / "X_ref_alt_marker_201.npy"
Y_PATH = PROCESSED_DIR / "y.npy"
METADATA_PATH = PROCESSED_DIR / "clinvar_training_metadata.parquet"

for path in [X_PATH, Y_PATH, METADATA_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

X_PATH, Y_PATH, METADATA_PATH

(PosixPath('/mnt/MyData/Bioinformatics/Project/processed/X_ref_alt_marker_201.npy'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/processed/y.npy'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/processed/clinvar_training_metadata.parquet'))

## 1. Cau hinh experiment

`RUN_FRACTION` cho phep chay nhanh tren mot phan train set. De train full dataset, dat `RUN_FRACTION = 1.0`.

In [2]:
MODEL_NAME = "transformer_small"  # cnn_baseline | cnn_residual | transformer_small
RUN_FRACTION = 1.0             # 1.0 = full train set; 0.1 = quick smoke test
EPOCHS = 5
BATCH_SIZE = 512
RANDOM_STATE = 42
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

MODEL_NAME, RUN_FRACTION, EPOCHS, BATCH_SIZE

('transformer_small', 1.0, 5, 512)

## 2. Load dataset

In [3]:
X = np.load(X_PATH, mmap_mode="r")
y = np.load(Y_PATH)
metadata_df = pd.read_parquet(METADATA_PATH)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"metadata shape: {metadata_df.shape}")
print(f"X dtype: {X.dtype}")
print(f"y dtype: {y.dtype}")

assert X.shape[1:] == (201, 9)
assert len(X) == len(y) == len(metadata_df)
assert np.all(X[:1000, :, 0:4].sum(axis=2) == 1)
assert np.all(X[:1000, :, 4:8].sum(axis=2) == 1)
assert np.all(X[:1000, 100, 8] == 1)

pd.Series(y).value_counts().rename(index={0: "Benign/Likely benign", 1: "Pathogenic/Likely pathogenic"})

X shape: (460488, 201, 9)
y shape: (460488,)
metadata shape: (460488, 15)
X dtype: uint8
y dtype: int8


Benign/Likely benign            401054
Pathogenic/Likely pathogenic     59434
Name: count, dtype: int64

## 3. Split train/val/test

In [4]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(y))
train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=RANDOM_STATE, stratify=y[temp_idx])

if RUN_FRACTION < 1.0:
    train_idx, _ = train_test_split(
        train_idx,
        train_size=RUN_FRACTION,
        random_state=RANDOM_STATE,
        stratify=y[train_idx],
    )

print(f"train: {len(train_idx):,}, labels={dict(zip(*np.unique(y[train_idx], return_counts=True)))}")
print(f"val:   {len(val_idx):,}, labels={dict(zip(*np.unique(y[val_idx], return_counts=True)))}")
print(f"test:  {len(test_idx):,}, labels={dict(zip(*np.unique(y[test_idx], return_counts=True)))}")

train: 322,341, labels={np.int8(0): np.int64(280737), np.int8(1): np.int64(41604)}
val:   69,073, labels={np.int8(0): np.int64(60158), np.int8(1): np.int64(8915)}
test:  69,074, labels={np.int8(0): np.int64(60159), np.int8(1): np.int64(8915)}


## 4. Dataset/DataLoader

In [5]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


class ClinVarSequenceDataset(Dataset):
    def __init__(self, X_array, y_array, indices):
        self.X_array = X_array
        self.y_array = y_array
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = self.indices[item]
        # Raw X: (201, 9). Conv1d/Transformer code uses (channels, length) = (9, 201).
        x_np = np.asarray(self.X_array[idx]).T.copy()
        x = torch.from_numpy(x_np).float()
        target = torch.tensor(self.y_array[idx], dtype=torch.float32)
        return x, target


train_loader = DataLoader(
    ClinVarSequenceDataset(X, y, train_idx),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    ClinVarSequenceDataset(X, y, val_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    ClinVarSequenceDataset(X, y, test_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

batch_x, batch_y = next(iter(train_loader))
print(batch_x.shape, batch_y.shape)

Device: cuda
torch.Size([512, 9, 201]) torch.Size([512])


## 5. Model zoo

In [6]:
class CNNBaseline(nn.Module):
    def __init__(self, in_channels=9):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


class ResidualBlock1D(nn.Module):
    def __init__(self, channels, kernel_size=5):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Conv1d(channels, channels, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(channels),
        )
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(x + self.block(x))


class CNNResidual(nn.Module):
    def __init__(self, in_channels=9):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 96, kernel_size=7, padding=3),
            nn.BatchNorm1d(96),
            nn.ReLU(),
        )
        self.blocks = nn.Sequential(
            ResidualBlock1D(96, kernel_size=5),
            nn.MaxPool1d(2),
            nn.Conv1d(96, 160, kernel_size=5, padding=2),
            nn.BatchNorm1d(160),
            nn.ReLU(),
            ResidualBlock1D(160, kernel_size=3),
            nn.AdaptiveMaxPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(160, 80),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(80, 1),
        )

    def forward(self, x):
        return self.classifier(self.blocks(self.stem(x))).squeeze(1)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=201):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TransformerSmall(nn.Module):
    def __init__(self, in_channels=9, d_model=96, nhead=4, num_layers=2, dim_feedforward=192):
        super().__init__()
        self.input_projection = nn.Linear(in_channels, d_model)
        self.position = PositionalEncoding(d_model=d_model, max_len=201)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=0.15,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(0.25),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # Input x: (batch, channels=9, length=201). Transformer uses (batch, length, channels).
        x = x.transpose(1, 2)
        x = self.input_projection(x)
        x = self.position(x)
        x = self.encoder(x)
        center_embedding = x[:, 100, :]
        return self.classifier(center_embedding).squeeze(1)


model_registry = {
    "cnn_baseline": CNNBaseline,
    "cnn_residual": CNNResidual,
    "transformer_small": TransformerSmall,
}

if MODEL_NAME not in model_registry:
    raise ValueError(f"Unknown MODEL_NAME={MODEL_NAME}. Options: {list(model_registry)}")

model = model_registry[MODEL_NAME]().to(DEVICE)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

TransformerSmall(
  (input_projection): Linear(in_features=9, out_features=96, bias=True)
  (position): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=96, out_features=96, bias=True)
        )
        (linear1): Linear(in_features=96, out_features=192, bias=True)
        (dropout): Dropout(p=0.15, inplace=False)
        (linear2): Linear(in_features=192, out_features=96, bias=True)
        (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.15, inplace=False)
        (dropout2): Dropout(p=0.15, inplace=False)
      )
    )
  )
  (classifier): Sequential(
    (0): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    (1): Dropout(p=0.25, inplace=False)
    (2): Linear(in_features=96, out_feature

/home/minato-one/.conda/envs/test-env/lib/python3.11/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


## 6. Train setup

In [7]:
# PyTorch/sympy compatibility guard for some notebook kernels.
import importlib
import sympy

if not hasattr(sympy, "core"):
    sympy.core = importlib.import_module("sympy.core")
if not hasattr(sympy.core, "symbol"):
    sympy.core.symbol = importlib.import_module("sympy.core.symbol")

positive_count = y[train_idx].sum()
negative_count = len(train_idx) - positive_count
pos_weight = torch.tensor([negative_count / max(positive_count, 1)], dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

print(f"pos_weight: {pos_weight.item():.4f}")

pos_weight: 6.7478


## 7. Train/evaluate loop

In [8]:
from sklearn.metrics import roc_auc_score, average_precision_score


def run_epoch(model, loader, train=False, epoch_label=""):
    model.train(train)
    total_loss = 0.0
    all_targets = []
    all_probs = []
    mode = "train" if train else "eval"

    progress = tqdm(loader, desc=f"{mode} {epoch_label}".strip(), unit="batch", leave=False)
    for batch_x, batch_y in progress:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)
        all_targets.append(batch_y.detach().cpu().numpy())
        progress.set_postfix(loss=f"{loss.item():.4f}")

    targets = np.concatenate(all_targets)
    probs = np.concatenate(all_probs)
    preds = (probs >= 0.5).astype(np.int8)
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": (preds == targets).mean(),
        "roc_auc": roc_auc_score(targets, probs),
        "pr_auc": average_precision_score(targets, probs),
    }, probs


best_val_auc = -np.inf
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS} - {MODEL_NAME}")
    train_metrics, _ = run_epoch(model, train_loader, train=True, epoch_label=f"{epoch}/{EPOCHS}")
    val_metrics, _ = run_epoch(model, val_loader, train=False, epoch_label=f"{epoch}/{EPOCHS}")
    scheduler.step(val_metrics["roc_auc"])

    row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

    if val_metrics["roc_auc"] > best_val_auc:
        best_val_auc = val_metrics["roc_auc"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"New best val_auc: {best_val_auc:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df

Epoch 1/5 - transformer_small


train 1/5:   0%|          | 0/630 [00:00<?, ?batch/s]

eval 1/5:   0%|          | 0/135 [00:00<?, ?batch/s]

{'epoch': 1, 'train_loss': 1.141, 'train_accuracy': np.float64(0.5929), 'train_roc_auc': 0.6539, 'train_pr_auc': 0.2229, 'val_loss': 1.0578, 'val_accuracy': np.float64(0.669), 'val_roc_auc': 0.7342, 'val_pr_auc': 0.3156}
New best val_auc: 0.7342
Epoch 2/5 - transformer_small


train 2/5:   0%|          | 0/630 [00:00<?, ?batch/s]

...

{'epoch': 4, 'train_loss': 0.9774, 'train_accuracy': np.float64(0.7268), 'train_roc_auc': 0.7837, 'train_pr_auc': 0.3868, 'val_loss': 0.9474, 'val_accuracy': np.float64(0.7138), 'val_roc_auc': 0.8017, 'val_pr_auc': 0.4298}
New best val_auc: 0.8017
Epoch 5/5 - transformer_small


train 5/5:   0%|          | 0/630 [00:00<?, ?batch/s]

eval 5/5:   0%|          | 0/135 [00:00<?, ?batch/s]

{'epoch': 5, 'train_loss': 0.9595, 'train_accuracy': np.float64(0.7349), 'train_roc_auc': 0.7935, 'train_pr_auc': 0.4052, 'val_loss': 0.9368, 'val_accuracy': np.float64(0.7436), 'val_roc_auc': 0.8053, 'val_pr_auc': 0.4429}
New best val_auc: 0.8053


,epoch,train_loss,train_accuracy,train_roc_auc,train_pr_auc,val_loss,val_accuracy,val_roc_auc,val_pr_auc
0,1,1.140960,0.592900,0.653850,0.222885,1.057752,0.668988,0.734248,0.315635
1,2,1.040202,0.686180,0.745121,0.317013,1.022618,0.670378,0.758813,0.330125
2,3,1.003640,0.712388,0.768692,0.355721,0.977346,0.731646,0.784420,0.393580
3,4,0.977356,0.726777,0.783663,0.386818,0.947425,0.713752,0.801693,0.429824
4,5,0.959452,0.734889,0.793529,0.405240,0.936759,0.743634,0.805250,0.442915


## 8. Test metrics

In [9]:
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

test_metrics, proba_test = run_epoch(model, test_loader, train=False, epoch_label="test")
pred_test = (proba_test >= 0.5).astype(np.int8)
y_test = y[test_idx]

print(test_metrics)
print("Classification report at threshold 0.5:")
print(classification_report(y_test, pred_test, target_names=["benign", "pathogenic"]))

confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, pred_test),
    index=["true_benign", "true_pathogenic"],
    columns=["pred_benign", "pred_pathogenic"],
)
confusion_matrix_df

eval test:   0%|          | 0/135 [00:00<?, ?batch/s]

{'loss': 0.9431660601928928, 'accuracy': np.float64(0.74410052986652), 'roc_auc': 0.8018347816871941, 'pr_auc': 0.4352754691491982}
Classification report at threshold 0.5:
              precision    recall  f1-score   support

      benign       0.95      0.75      0.84     60159
  pathogenic       0.30      0.71      0.42      8915

    accuracy                           0.74     69074
   macro avg       0.62      0.73      0.63     69074
weighted avg       0.86      0.74      0.78     69074



,pred_benign,pred_pathogenic
true_benign,45071,15088
true_pathogenic,2588,6327


## 8B. Danh gia nhanh ket qua transformer_small

Ket qua `transformer_small` tren full dataset 9-channel:

- Test ROC-AUC: `~0.802`
- Test PR-AUC: `~0.435`
- Accuracy tai threshold `0.5`: `~0.744`
- Pathogenic precision tai threshold `0.5`: `~0.30`
- Pathogenic recall tai threshold `0.5`: `~0.71`
- F1 pathogenic tai threshold `0.5`: `~0.42`

Confusion matrix tai threshold `0.5`:

- True benign du doan dung: `45,071`
- True benign bi goi nham pathogenic: `15,088`
- True pathogenic bi bo sot: `2,588`
- True pathogenic du doan dung: `6,327`

Threshold analysis:

- `Recall at precision >= 0.50`: `~0.356`
- `Precision at recall >= 0.80`: `~0.248`
- Best F1: threshold `~0.652`, precision `~0.398`, recall `~0.519`, F1 `~0.451`
- Best F2: threshold `~0.468`, precision `~0.280`, recall `~0.742`, F2 `~0.558`

So voi CNN 9-channel o notebook 02:

- CNN test ROC-AUC `~0.848` > Transformer `~0.802`
- CNN test PR-AUC `~0.522` > Transformer `~0.435`
- CNN pathogenic precision `~0.35` > Transformer `~0.30`
- CNN pathogenic recall `~0.72` gan bang Transformer `~0.71`
- CNN false positive benign->pathogenic `11,809` < Transformer `15,088`

Ket luan:

Transformer demo nay chay duoc va hoc duoc tin hieu, nhung **chua tot bang CNN 9-channel baseline**. Với chuoi ngan `201 bp`, CNN co inductive bias cuc bo tot hon va it tham so hon nen dang phu hop hon. Transformer co the can tuning them: d_model/layers, learning rate, warmup, batch size, pooling strategy, va train lau hon. Hien tai nen giu CNN 9-channel lam baseline chinh, dung transformer nhu nhanh thu nghiem.

## 9. Threshold analysis

Tinh cac chi so de chon threshold thuc dung hon `0.5`.

In [11]:
precision, recall, thresholds = precision_recall_curve(y_test, proba_test)

# precision/recall arrays have one extra point compared with thresholds.
pr_table = pd.DataFrame({
    "threshold": np.r_[thresholds, 1.0],
    "precision": precision,
    "recall": recall,
})

recall_at_precision_50 = pr_table.loc[pr_table["precision"].ge(0.50), "recall"].max()
precision_at_recall_80 = pr_table.loc[pr_table["recall"].ge(0.80), "precision"].max()

f1 = 2 * pr_table["precision"] * pr_table["recall"] / (pr_table["precision"] + pr_table["recall"] + 1e-12)
f2 = 5 * pr_table["precision"] * pr_table["recall"] / (4 * pr_table["precision"] + pr_table["recall"] + 1e-12)

best_f1_row = pr_table.iloc[f1.idxmax()].copy()
best_f1_row["f1"] = f1.max()
best_f2_row = pr_table.iloc[f2.idxmax()].copy()
best_f2_row["f2"] = f2.max()

print(f"Recall at precision >= 0.50: {recall_at_precision_50:.4f}")
print(f"Precision at recall >= 0.80: {precision_at_recall_80:.4f}")
print("Best F1 threshold:")
display(best_f1_row.to_frame().T)
print("Best F2 threshold:")
display(best_f2_row.to_frame().T)

pr_table.iloc[:: max(len(pr_table) // 20, 1)].head(25)

Recall at precision >= 0.50: 0.3556
Precision at recall >= 0.80: 0.2484
Best F1 threshold:


,threshold,precision,recall,f1
57326,0.652238,0.398364,0.519013,0.450755


Best F2 threshold:


,threshold,precision,recall,f2
45338,0.468479,0.280027,0.742457,0.558123


,threshold,precision,recall
0,0.060237,0.129064,1.000000
3446,0.116901,0.135192,0.995177
6892,0.142069,0.141495,0.986764
10338,0.165607,0.148241,0.976444
13784,0.188656,0.155962,0.966910
17230,0.211631,0.164376,0.955356
20676,0.235481,0.172937,0.937970
24122,0.261455,0.182731,0.920359
27568,0.289063,0.193404,0.899159
31014,0.317898,0.205632,0.876388


## 10. Save experiment outputs

In [12]:
safe_model_name = MODEL_NAME.replace("/", "_")
model_path = MODEL_DIR / f"clinvar_{safe_model_name}_pytorch.pt"
prediction_path = PROCESSED_DIR / f"{safe_model_name}_test_predictions.parquet"
history_path = PROCESSED_DIR / f"{safe_model_name}_training_history.parquet"
threshold_path = PROCESSED_DIR / f"{safe_model_name}_threshold_table.parquet"

metadata_out = metadata_df.iloc[test_idx].copy()
metadata_out["pred_proba_pathogenic"] = proba_test
metadata_out["pred_label_05"] = pred_test

torch.save(
    {
        "model_name": MODEL_NAME,
        "model_state_dict": model.state_dict(),
        "input_shape": (9, 201),
        "channels": ["ref_A", "ref_C", "ref_G", "ref_T", "alt_A", "alt_C", "alt_G", "alt_T", "mutation_marker"],
        "test_metrics": test_metrics,
        "config": {
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "run_fraction": RUN_FRACTION,
            "random_state": RANDOM_STATE,
        },
    },
    model_path,
)
metadata_out.to_parquet(prediction_path, index=False, engine="pyarrow")
history_df.to_parquet(history_path, index=False, engine="pyarrow")
pr_table.to_parquet(threshold_path, index=False, engine="pyarrow")

print(f"Saved model: {model_path}")
print(f"Saved predictions: {prediction_path}")
print(f"Saved history: {history_path}")
print(f"Saved threshold table: {threshold_path}")

Saved model: /mnt/MyData/Bioinformatics/Project/models/clinvar_transformer_small_pytorch.pt
Saved predictions: /mnt/MyData/Bioinformatics/Project/processed/transformer_small_test_predictions.parquet
Saved history: /mnt/MyData/Bioinformatics/Project/processed/transformer_small_training_history.parquet
Saved threshold table: /mnt/MyData/Bioinformatics/Project/processed/transformer_small_threshold_table.parquet
